In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.models.smoke_test import make_gaussian, make_banana, make_gaussian_mixture, plot_marginals
from sazz.utils.sampling import resample_pdmp_path

In [ ]:
def run_test(target, thinning, N_skeleton=50_000, N_resample=50_000,
             refresh_rate=1.0, burnin_frac=0.1, pli_kwargs=None):
    """
    Build sampler, run, resample, return numpy samples.
    """

    sampler = AutomaticBoomerangSampler(
        grad_target=target.grad_target,
        D=target.D,
        refresh_rate=refresh_rate,
        thinning=thinning,
        pli_kwargs=pli_kwargs,
    )
    sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

    result = sampler.sample(N=N_skeleton, diagnostics=True)

    # Convert to numpy for resampling
    pos_np = result["positions"].cpu().numpy()
    vel_np = result["velocities"].cpu().numpy()
    tim_np = result["times"].cpu().numpy()
    x_ref_np = target.x_ref.cpu().numpy()

    samples = resample_pdmp_path(pos_np, vel_np, tim_np, x_ref_np,
                                 N_resample, burnin_frac=burnin_frac)
    return samples

In [ ]:
N_SKELETON = 10_000
N_RESAMPLE = 100_000

# ── 1. Gaussian (should be trivially correct with matched reference) ──
print("\n" + "="*60)
print("TEST 1: Diagonal Gaussian D=5  (matched reference)")
print("="*60)
target_gauss = make_gaussian(D=5, cov="diagonal")

samp_brent = run_test(target_gauss, thinning="brent",
                        N_skeleton=N_SKELETON, N_resample=N_RESAMPLE)
samp_pli = run_test(target_gauss, thinning="pli",
                    N_skeleton=N_SKELETON, N_resample=N_RESAMPLE)

plot_marginals(target_gauss, {"Brent": samp_brent, "PLI": samp_pli})

In [ ]:
# ── 2. AR(1) Gaussian ──
print("\n" + "="*60)
print("TEST 2: AR(1) Gaussian D=5")
print("="*60)
target_ar = make_gaussian(D=5, cov="ar1")

samp_brent_ar = run_test(target_ar, thinning="brent",
                        N_skeleton=N_SKELETON, N_resample=N_RESAMPLE)

samp_pli_ar = run_test(target_ar, thinning="pli",
                        N_skeleton=N_SKELETON, N_resample=N_RESAMPLE)
plot_marginals(target_ar, {"Brent": samp_brent_ar, "PLI": samp_pli_ar})

In [ ]:
# ── 3. Bimodal mixture D=1 ──
print("\n" + "="*60)
print("TEST 3: Bimodal mixture D=1")
print("="*60)
target_mix = make_gaussian_mixture(D=1, preset="bimodal")

samp_mix_brent = run_test(target_mix, thinning="brent",
                    N_skeleton=N_SKELETON, N_resample=N_RESAMPLE,
                    refresh_rate=1.0)

samp_mix_pli = run_test(target_mix, thinning="pli",
                    N_skeleton=N_SKELETON, N_resample=N_RESAMPLE,
                    refresh_rate=1.0)
plot_marginals(target_mix, {"Brent":samp_mix_brent, "PLI": samp_mix_pli})

In [ ]:
# ── 4. Banana D=2 ──
print("\n" + "="*60)
print("TEST 4: Banana D=2, a=1, scale=1")
print("="*60)
target_ban = make_banana(D=2, a=1.0, scale=1.0)

samp_ban_brent = run_test(target_ban, thinning="brent",
                    N_skeleton=N_SKELETON, N_resample=N_RESAMPLE,
                    refresh_rate=1.0)

samp_ban_pli = run_test(target_ban, thinning="pli",
                    N_skeleton=N_SKELETON, N_resample=N_RESAMPLE,
                    refresh_rate=1.0)
plot_marginals(target_ban, {"Brent":samp_ban_brent, "PLI": samp_ban_pli})

print("\nAll smoke tests complete.")

## TEST WARMUP

In [ ]:
from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.utils.warmup import warmup
from sazz.utils.sampling import resample_pdmp_path
from sazz.models.smoke_test import make_banana

N_SKELETON = 10_000
N_RESAMPLE = 100_000

print("\n" + "=" * 60)
print("TEST: Banana D=2, a=1, scale=1")
print("=" * 60)

target_ban = make_banana(D=2, a=1.0, scale=1.0)

# --- Brent ---
sampler_brent = AutomaticBoomerangSampler(
    grad_target=target_ban.grad_target,
    D=target_ban.D,
    refresh_rate=0.1,
    thinning="brent",
)
warmup(sampler_brent, n_rounds=3, n_pilot=500, target=target_ban)
result_brent = sampler_brent.sample(N=N_SKELETON, diagnostics=True)

samples_brent = resample_pdmp_path(
    result_brent["positions"].cpu().numpy(),
    result_brent["velocities"].cpu().numpy(),
    result_brent["times"].cpu().numpy(),
    sampler_brent.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
)

# --- PLI ---
sampler_pli = AutomaticBoomerangSampler(
    grad_target=target_ban.grad_target,
    D=target_ban.D,
    refresh_rate=0.1,
    thinning="pli",
)
warmup(sampler_pli, n_rounds=3, n_pilot=500, target=target_ban)
result_pli = sampler_pli.sample(N=N_SKELETON, diagnostics=True)

samples_pli = resample_pdmp_path(
    result_pli["positions"].cpu().numpy(),
    result_pli["velocities"].cpu().numpy(),
    result_pli["times"].cpu().numpy(),
    sampler_pli.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE,
)

# --- Plot ---
plot_marginals(target_ban, {"Brent": samples_brent, "PLI": samples_pli})